In [ ]:
# Importation des packages et réglagle de la seed pour fixé un nombre de simulation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy_financial as npf

SEED = 2107
rng = np.random.default_rng(SEED)


In [ ]:
# Paramètre de base pour les calcules

#Horizon temps
an_debut = 2027
an_fin = 2060
n_ans = an_fin - an_debut +1
cible = 3500000

#Revenue & dépense
salaire_base = 95000
Div_inc_base = 0
hypo_FIXE = 38900
fin_hypo = 2057
depense_base = 38000
fct_croissance_depense = 0.03
taux_depense_compressible = 0.12
taux_remb_dette = 0.20

#Bilan personnel au 31 Dec 2026
placement_base = 45000
propriete = 920000
inc_valeur_revente = 0
hypo_solde = 598000
autre_dette = 0

# Vérification des valeurs
valeur_nette_base = placement_base + propriete - hypo_solde - autre_dette
surplus_2026 = salaire_base - hypo_FIXE - depense_base - taux_depense_compressible * salaire_base

print("Nombre d'années:", n_ans)
print("Valeur nette:", valeur_nette_base)
print("Surplus 2026:", surplus_2026)

Nombre d'années: 34
Valeur nette: 367000
Surplus 2026: 6700.0


In [ ]:
# Étape --- Hypothèque 

# Trouver le taux d'hypotèque
n_versements = fin_hypo - an_debut + 1

taux_hypo = npf.rate(nper=n_versements, pmt=-hypo_FIXE, pv=hypo_solde, fv=0)
print("taux Hypothèque :", round(taux_hypo,4))

# Faire le tableau d'amortissement
soldes = []
solde = hypo_solde
for annee in range(an_debut, an_fin + 1):
    if annee <= fin_hypo:
        interet = solde * taux_hypo
        solde = solde - (hypo_FIXE - interet)
    soldes.append(max(solde, 0))

soldes = np.array(soldes)
print("Solde en 2057 :", round(soldes[fin_hypo - an_debut],2))

taux Hypothèque : 0.0512
Solde en 2057 : 0.0


In [ ]:
# Étape --- Modélisation des Taux d'intérêt (bons du trésor)

taux_base = 0.0310
n_trimestre = n_ans *4

def simuler_taux(n_scenario, rng):
    taux = np.empty((n_scenario, n_trimestre))
    r = np.full(n_scenario, taux_base)

    for t in range(n_trimestre):
        z = rng.standard_normal(n_scenario)
        r = r + 0.1 * (0.03 - r) + 0.02 * np.sqrt(np.maximum(r, 0)) * z
        taux[:, t] = np.maximum(r, 0)

    return taux

# Vérification des résultats
taux_test = simuler_taux(2500, rng)
print(taux_test.shape)
print("taux min: ", round(taux_test.min(), 4))
print("Moyenne en 2060", round(taux_test[:, -1].mean(), 4))

### Entre 2.9% et 3.1% selon les simulations ###

(2500, 136)
taux min:  0.0064
Moyenne en 2060 0.03
